In [1]:
import sys
import os

sys.path.append('/scratch2/mrenaudin/colorlessgreenRNNs')

In [2]:
from src.language_models import model as m
import torch
from utils import NounPPDataset, collate_fn_nounpp
from src.language_models.dictionary_corpus import Dictionary
from torch.utils.data import DataLoader
from collections import defaultdict



/home/mrenaudin/.conda/envs/leaps3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
device = torch.device('cpu')
model = m.CBR_RNN(50001, 1024, 1024, 1, 0, device)
checkpoint = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/test_attention_1024_rest/epoch_40.pt', map_location='cpu')
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
dictionary = Dictionary(data_path)
nounpp = "//scratch2/mrenaudin/colorlessgreenRNNs/NounPP/Stimuli/nounpp.txt"


In [18]:
test_dataset = NounPPDataset(nounpp, dictionary)
test_dataloader = DataLoader(test_dataset, batch_size=1024, collate_fn=collate_fn_nounpp)

In [19]:
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [20]:
temp = checkpoint['temperature'] #that's because of mistake in save checkpoints function

In [21]:
checkpoint

{'epoch': 40,
 'model_state_dict': OrderedDict([('encoder.weight',
               tensor([[ 0.1750,  0.3779,  0.0781,  ..., -0.0444, -0.2846, -0.0384],
                       [-0.2353,  0.2754, -0.3836,  ...,  0.1782,  0.0543,  0.0746],
                       [-0.1380,  0.0485, -0.0444,  ..., -0.0309,  0.1485,  0.1641],
                       ...,
                       [-0.2668,  0.1855,  0.1279,  ...,  0.0152, -0.2186, -0.1160],
                       [-0.0635, -0.1848,  0.0893,  ...,  0.1379, -0.0635,  0.4351],
                       [-0.1688,  0.0416,  0.2592,  ..., -0.1474,  0.0867,  0.0939]])),
              ('q.weight',
               tensor([[-0.1073,  0.0185, -0.1315,  ...,  0.0768,  0.1468,  0.5339],
                       [ 0.0151,  0.0650,  0.1173,  ...,  0.2539,  0.0787,  0.2378],
                       [-0.0367,  0.1169,  0.0280,  ..., -0.0863,  0.0596,  0.1339],
                       ...,
                       [-0.4349, -0.9149, -0.1393,  ...,  0.3874, -0.0984, -0.0421

In [22]:
def eval(model, test_dataloader, temperature):
    condition_accuracies = defaultdict(int)
    condition_counts = defaultdict(int)
    correct_pred = 0
    sentence_details = []
    model.eval()
    # Forward pass with hidden state update word by word
    with torch.no_grad():
        for batch in test_dataloader:
            out = None
            written = batch["sentence"]
            sentence = batch["encoded_sentence"]
            correct = batch["encoded_correct"]
            wrong = batch["encoded_wrong"]
            condition = batch["condition"]
            batch_size = sentence.size(0)

            sent = sentence[:, :5].transpose(0, 1)
            cache = model.init_cache(sent,1)  # regarder si on peut mettre du priming
            # for i in range(sent.shape[1]):
            out, cache = model(sent, cache, 1, temperature, True)
            log_probs = torch.nn.functional.log_softmax(
                out, dim=-1
            )  # s(out.squeeze(0))
            # déja sur correct et wrong log probs, pas les même résultats que sur extract_predictions.py
            correct_log_probs = log_probs[
                -1, torch.arange(batch_size), correct
            ]  # Shape: [512]
            wrong_log_probs = log_probs[-1, torch.arange(batch_size), wrong]
            correct_predictions = correct_log_probs >= wrong_log_probs

            for i in range(batch_size):
                cond = condition[i]
                pred = correct_predictions[i].item()  # Convert tensor to Python boolean
                condition_counts[cond] += 1
                condition_accuracies[cond] += pred

                sentence_details.append(
                    {
                        "sentence": written[i],
                        "condition": condition[i],
                        "correct_log_prob": correct_log_probs[i],
                        "wrong_log_prob": wrong_log_probs[i],
                        "model_prefers_correct": pred,
                    }
                )

    final_accuracies = {
        cond: condition_accuracies[cond] / condition_counts[cond]
        for cond in condition_accuracies
    }
    return final_accuracies

In [23]:
eval(model, test_dataloader, temp)

{'singular singular': 0.734,
 'singular plural': 0.495,
 'plural singular': 0.936,
 'plural plural': 0.901}